# Lesson 5: Event-driven generation

### Import all needed packages

In [3]:
import boto3, os

from helpers.Lambda_Helper import Lambda_Helper
from helpers.S3_Helper import S3_Helper

lambda_helper = Lambda_Helper()
s3_helper = S3_Helper()

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [4]:
bucket_name_text = os.getenv('LEARNERS3BUCKETNAMETEXT')
print(bucket_name_text)
bucket_name_audio = os.getenv('LEARNERS3BUCKETNAMEAUDIO')
print(bucket_name_audio)

TODO
TODO


### Deploy your lambda function

In [8]:
%%writefile lambda_function.py

#############################################################
#
# This Lambda function is written to a file by the notebook 
# It does not run in the notebook!
#
#############################################################

import json
import boto3
import uuid
import os

s3_client = boto3.client('s3')
transcribe_client = boto3.client('transcribe', region_name='us-west-2')

def lambda_handler(event, context):
    # Extract the bucket name and key from the incoming event
    bucket = event['Records'][0]['s3']['bucket']['name']
    key = event['Records'][0]['s3']['object']['key']

    # One of a few different checks to ensure we don't end up in a recursive loop.
    if key != "dialog.mp3": 
        print("This demo only works with dialog.mp3.")
        return

    try:
        
        job_name = 'transcription-job-' + str(uuid.uuid4()) # Needs to be a unique name

        response = transcribe_client.start_transcription_job(
            TranscriptionJobName=job_name,
            Media={'MediaFileUri': f's3://{bucket}/{key}'},
            MediaFormat='mp3',
            LanguageCode='en-US',
            OutputBucketName= os.environ['S3BUCKETNAMETEXT'],  # specify the output bucket
            OutputKey=f'{job_name}-transcript.json',
            Settings={
                'ShowSpeakerLabels': True,
                'MaxSpeakerLabels': 2
            }
        )
        
    except Exception as e:
        print(f"Error occurred: {e}")
        return {
            'statusCode': 500,
            'body': json.dumps(f"Error occurred: {e}")
        }

    return {
        'statusCode': 200,
        'body': json.dumps(f"Submitted transcription job for {key} from bucket {bucket}.")
    }



Overwriting lambda_function.py


In [6]:
lambda_helper.lambda_environ_variables = {'S3BUCKETNAMETEXT' : bucket_name_text}
lambda_helper.deploy_function(["lambda_function.py"], function_name="LambdaFunctionTranscribe")

Zipping function...
Looking for existing function...
Function LambdaFunctionTranscribe does not exist. Creating...


KeyError: 'LAMBDALAYERVERSIONARN'

In [7]:
lambda_helper.filter_rules_suffix = "mp3"
lambda_helper.add_lambda_trigger(bucket_name_audio, function_name="LambdaFunctionTranscribe")

Error adding Lambda permission: An error occurred (ResourceNotFoundException) when calling the AddPermission operation: Function not found: arn:aws:lambda:us-west-2:092413168457:function:LambdaFunctionTranscribe


In [ ]:
s3_helper.upload_file(bucket_name_audio, 'dialog.mp3')

In [ ]:
s3_helper.list_objects(bucket_name_audio)

#### Restart kernel if needed.
- If you run the code fairly quickly from start to finish, it's possible that the following code cell `s3_helper.list_objects(bucket_name_text)` will give a "Not Found" error.  
- If waiting a few seconds (10 seconds) and re-running this cell does not resolve the error, then you can restart the kernel of the jupyter notebook.
- Go to menu->Kernel->Restart Kernel.
- Then run the code cells from the start of the notebook, waiting 2 seconds or so for each code cell to finish executing.

In [ ]:
s3_helper.list_objects(bucket_name_text)

#### Re-run "download" code cell as needed
- It may take a few seconds for the results to be generated.
- If you see a `Not Found` error, please wait a few seconds and then try running the `s3_helper.download_object` again.

In [ ]:
s3_helper.download_object(bucket_name_text, 'results.txt')

In [ ]:
from helpers.Display_Helper import Display_Helper


In [ ]:
display_helper = Display_Helper()


In [ ]:
display_helper.text_file('results.txt')

Extra resources:

* [Generative AI code](https://community.aws/code/generative-ai)

* [Generative AI](https://community.aws/generative-ai)
